# Notebook 1: Summarization Model Exploration

Compare different pre-trained summarization models on cybersecurity news articles.

**Models tested:**
- facebook/bart-large-cnn
- sshleifer/distilbart-cnn-6-6
- google/pegasus-cnn_dailymail
- allenai/led-base-16384
- falconsai/text_summarization (T5-small)

In [ ]:
!pip install transformers torch rouge-score datasets accelerate -q

In [ ]:
import time
import json
from pathlib import Path
from transformers import pipeline
from rouge_score import rouge_scorer

MODELS = {
    "bart-large-cnn": "facebook/bart-large-cnn",
    "distilbart-cnn-6-6": "sshleifer/distilbart-cnn-6-6",
    "pegasus-cnn_dailymail": "google/pegasus-cnn_dailymail",
    "led-base-16384": "allenai/led-base-16384",
    "t5-small-summary": "falconsai/text_summarization",
}

SAMPLE_ARTICLES = [
    {
        "article": """A critical vulnerability in Microsoft Exchange Server, tracked as CVE-2024-12345, is being actively exploited by a Chinese threat group dubbed Storm-0558. The flaw allows remote code execution without authentication, giving attackers full access to email accounts and administrative privileges. Microsoft released an out-of-band security update on Tuesday, urging all Exchange Server administrators to patch immediately. The company confirmed that at least 30 organizations in the United States have been compromised, including government agencies and defense contractors. CISA has added the vulnerability to its Known Exploited Vulnerabilities catalog, requiring federal agencies to patch within 21 days. Security researchers from Volexity, who discovered the exploitation in early January, noted that the attackers used a novel technique to bypass authentication by forging Kerberos tokens using a stolen signing key.""",
        "reference": "Microsoft Exchange Server vulnerability CVE-2024-12345 is being actively exploited by Chinese threat group Storm-0558, allowing unauthenticated remote code execution. At least 30 US organizations have been compromised. Microsoft released an emergency patch and CISA added the flaw to its Known Exploited Vulnerabilities catalog.",
    },
    {
        "article": """Ransomware attacks on healthcare organizations surged 94% in 2023 compared to the previous year, according to a new report from Sophos. The average ransom demand reached $1.5 million, while the average cost of recovery exceeded $3.4 million including downtime, legal fees, and regulatory fines. The report surveyed 500 healthcare IT professionals across 14 countries and found that 67% of organizations hit by ransomware paid the ransom, though only 58% of those recovered all their data. LockBit and BlackCat were the most prevalent ransomware families targeting healthcare. Sophos recommends that healthcare organizations implement zero-trust architecture, maintain offline backups, and conduct regular incident response drills to mitigate the growing threat.""",
        "reference": "Ransomware attacks on healthcare rose 94% in 2023, with average ransom demands of $1.5M and recovery costs exceeding $3.4M. 67% of victims paid ransoms, but only 58% recovered all data. LockBit and BlackCat were the most common ransomware families targeting healthcare.",
    },
    {
        "article": """The Cybersecurity and Infrastructure Security Agency (CISA) on Thursday published new binding operational directives requiring all federal civilian agencies to adopt multifactor authentication and patch known vulnerabilities within strict timelines. Directive BOD 23-02 mandates that agencies implement phishing-resistant MFA for all staff and contractor accounts within 180 days, while BOD 23-01 requires patching of critical vulnerabilities within 14 days and high-severity flaws within 30 days. Agencies that fail to comply must submit remediation plans to CISA. The directives follow a series of high-profile breaches including the SolarWinds supply chain attack and the Log4Shell vulnerability exploitation. CISA Director Jen Easterly stated that the new requirements represent a fundamental shift in how the federal government approaches cybersecurity risk management.""",
        "reference": "CISA issued binding operational directives requiring federal agencies to implement phishing-resistant MFA within 180 days and patch critical vulnerabilities within 14 days. Non-compliant agencies must submit remediation plans. The directives follow major breaches like SolarWinds and Log4Shell.",
    },
]

print(f"Loaded {len(SAMPLE_ARTICLES)} sample articles")

## Run All Models

In [ ]:
results = {}

for model_name, model_id in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Loading {model_name} ({model_id})...")
    
    try:
        start = time.time()
        summarizer = pipeline("summarization", model=model_id, device_map="auto")
        load_time = time.time() - start
        print(f"  Loaded in {load_time:.1f}s")
        
        model_results = []
        for i, sample in enumerate(SAMPLE_ARTICLES):
            text = sample["article"]
            
            try:
                start = time.time()
                output = summarizer(text, max_length=150, min_length=30, do_sample=False)
                infer_time = time.time() - start
                summary = output[0]["summary_text"]
                error = None
            except Exception as e:
                summary = ""
                infer_time = 0
                error = str(e)
            
            model_results.append({
                "article_id": i,
                "summary": summary,
                "inference_time": round(infer_time, 2),
                "error": error,
                "length": len(summary.split()),
            })
            status = f"OK ({infer_time:.1f}s, {len(summary.split())} words)" if not error else f"ERROR: {error[:80]}"
            print(f"  Article {i}: {status}")
        
        results[model_name] = {
            "model_id": model_id,
            "load_time": round(load_time, 1),
            "articles": model_results,
        }
        
    except Exception as e:
        print(f"  FAILED: {e}")
        results[model_name] = {"model_id": model_id, "error": str(e)}
        continue

    del summarizer
    import gc, torch
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nDone! Evaluated {len(results)} models.")

## Compute ROUGE Scores

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

print(f"{'Model':<25} {'ROUGE-1':>10} {'ROUGE-2':>10} {'ROUGE-L':>10} {'Avg Time':>10}")
print("-" * 70)

comparison = []
for model_name, data in results.items():
    if "error" in data:
        print(f"{model_name:<25} {'FAILED':>10}")
        continue
    
    r1_scores, r2_scores, rl_scores = [], [], []
    times = []
    
    for i, res in enumerate(data["articles"]):
        if res.get("error"):
            continue
        reference = SAMPLE_ARTICLES[i]["reference"]
        scores = scorer.score(reference, res["summary"])
        r1_scores.append(scores["rouge1"].fmeasure)
        r2_scores.append(scores["rouge2"].fmeasure)
        rl_scores.append(scores["rougeL"].fmeasure)
        times.append(res["inference_time"])
    
    avg_r1 = sum(r1_scores) / len(r1_scores) if r1_scores else 0
    avg_r2 = sum(r2_scores) / len(r2_scores) if r2_scores else 0
    avg_rl = sum(rl_scores) / len(rl_scores) if rl_scores else 0
    avg_time = sum(times) / len(times) if times else 0
    
    print(f"{model_name:<25} {avg_r1:>10.4f} {avg_r2:>10.4f} {avg_rl:>10.4f} {avg_time:>9.2f}s")
    
    comparison.append({
        "model": model_name,
        "model_id": data["model_id"],
        "rouge1": round(avg_r1, 4),
        "rouge2": round(avg_r2, 4),
        "rougeL": round(avg_rl, 4),
        "avg_inference_time": round(avg_time, 2),
        "load_time": data["load_time"],
    })

with open("model_comparison_results.json", "w") as f:
    json.dump(comparison, f, indent=2)
print("\nResults saved to model_comparison_results.json")

## Show Sample Summaries Side-by-Side

In [ ]:
for i, sample in enumerate(SAMPLE_ARTICLES):
    print(f"\n{'='*80}")
    print(f"ARTICLE {i+1} (first 200 chars): {sample['article'][:200]}...")
    print(f"REFERENCE: {sample['reference']}")
    print(f"{'-'*80}")
    
    for model_name, data in results.items():
        if "error" in data:
            continue
        res = data["articles"][i]
        if res.get("error"):
            print(f"  {model_name}: ERROR - {res['error'][:60]}")
        else:
            print(f"  {model_name}: {res['summary']}")
    print()

## Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

output_dir = Path('/content/drive/MyDrive/model-lab/exploration')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "model_comparison_results.json", "w") as f:
    json.dump(comparison, f, indent=2)

for model_name, data in results.items():
    if "error" in data:
        continue
    for i, res in enumerate(data["articles"]):
        res["reference"] = SAMPLE_ARTICLES[i]["reference"]
    with open(output_dir / f"{model_name}_summaries.json", "w") as f:
        json.dump(data, f, indent=2)

print(f"Saved to {output_dir}")